# 01 · Data catalog: what we have and what it means

Everything in the database, explained. Reads `data/gbc.duckdb` (no PostgreSQL needed).

**The project question:** *how do the social and economic conditions of a country relate to the performance of the companies that operate there?*
So there are two kinds of data, and the project depends on joining them:

| | What it is | Role | Tables |
|---|---|---|---|
| **Country data** | Economic, social, institutional and business indicators per country per year | the "conditions" (explanatory side) | `dim_country`, `dim_indicator`, `fact_country_year` |
| **Company data** | Annual financial statements of listed companies | the "performance" (outcome side) | `dim_company`, `fact_company_financial`, `v_company_year` |

> This data can show **association**, not causation. A company's results depend mostly on the firm and its sector; country conditions explain a smaller part.

**Contents:** 1 Sources · 2 Countries · 3 Country indicators (catalog, coverage, snapshot, trends, a data break) · 4 Companies · 5 Quality notes · 6 Data dictionary

In [1]:
import pathlib
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
con = duckdb.connect(str(ROOT / "data" / "gbc.duckdb"), read_only=True)

def q(sql):
    return con.sql(sql).df()

pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 200,
              "display.max_colwidth", 110, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"
print("database:", ROOT / "data" / "gbc.duckdb")

database: /Users/hugo/Projects/name_to_decide/data/gbc.duckdb


## 1 · Sources
Six sources, all free and without an API key. Each one is downloaded once and kept untouched in `data/raw/`.

In [2]:
SOURCE_INFO = {
    "WB":   ("Country indicators on economy, population, labour, finance, trade, education and infrastructure.", "All 34 countries, 2012–2024 (1–2 year publication lag)"),
    "WGI":  ("Six institutional-quality scores (rule of law, corruption control, ...) built from surveys and expert ratings.", "All 34 countries, 2012–2024"),
    "IMF":  ("Macro indicators from the World Economic Outlook: growth, inflation, unemployment, government debt, current account. Projections after 2024 were dropped.", "All 34 countries, 2012–2024"),
    "ESTAT":("EU-only statistics: labour cost, minimum wage, digital adoption, R&D, business structure and demography, corporate profitability.", "21 EU countries only; periods differ by dataset"),
    "YF":   ("Annual income statements and balance sheets of listed companies scraped from Yahoo Finance (unofficial; may break).", "~2,590 companies, 34 countries, about 4 fiscal years (2022–2025)"),
    "ESEF": ("IFRS annual reports that EU listed companies must file in machine-readable form (filings.xbrl.org).", "~1,310 companies, 20 EU countries (not Germany, not Ireland), fiscal years 2019–2025"),
}
src = q("""
SELECT s.source_id, s.name, s.base_url, s.retrieved_at::date AS retrieved,
       (SELECT COUNT(*) FROM dim_indicator i WHERE i.source_id = s.source_id) AS indicators,
       (SELECT COUNT(*) FROM fact_country_year f JOIN dim_indicator i USING (indicator_id) WHERE i.source_id = s.source_id) AS country_year_rows,
       (SELECT COUNT(*) FROM dim_company c WHERE c.source_id = s.source_id) AS companies
FROM dim_source s ORDER BY 1
""")
src["what_it_gives_us"] = src.source_id.map(lambda k: SOURCE_INFO[k][0])
src["coverage"] = src.source_id.map(lambda k: SOURCE_INFO[k][1])
src.drop(columns=["base_url"])

,source_id,name,retrieved,indicators,country_year_rows,companies,what_it_gives_us,coverage
0,ESEF,ESEF filings (filings.xbrl.org),2026-10-05,0,0,1312,IFRS annual reports that EU listed companies must file in machine-readable form (filings.xbrl.org).,"~1,310 companies, 20 EU countries (not Germany, not Ireland), fiscal years 2019–2025"
1,ESTAT,Eurostat,2026-10-05,34,6930,0,"EU-only statistics: labour cost, minimum wage, digital adoption, R&D, business structure and demography, c...",21 EU countries only; periods differ by dataset
2,IMF,IMF DataMapper (World Economic Outlook),2026-10-05,6,2641,0,"Macro indicators from the World Economic Outlook: growth, inflation, unemployment, government debt, curren...","All 34 countries, 2012–2024"
3,WB,World Bank WDI,2026-10-05,31,12695,0,"Country indicators on economy, population, labour, finance, trade, education and infrastructure.","All 34 countries, 2012–2024 (1–2 year publication lag)"
4,WGI,Worldwide Governance Indicators (World Bank source 3),2026-10-05,6,2652,0,"Six institutional-quality scores (rule of law, corruption control, ...) built from surveys and expert rati...","All 34 countries, 2012–2024"
5,YF,Yahoo Finance via yfinance,2026-10-05,0,0,2590,Annual income statements and balance sheets of listed companies scraped from Yahoo Finance (unofficial; ma...,"~2,590 companies, 34 countries, about 4 fiscal years (2022–2025)"


## 2 · Countries (`dim_country`)
34 countries: 21 EU members and 13 outside Europe, chosen to mix regions and income levels.
Five EU members with no usable company data (Bulgaria, Croatia, Cyprus, Slovenia, Slovakia) and Malta (one company) were dropped.

| Column | Meaning |
|---|---|
| `iso3` | Three-letter code; **the key used everywhere** (World Bank, IMF, our joins) |
| `iso2`, `eurostat_code` | Other code systems (Eurostat uses `EL` for Greece, and only covers the EU) |
| `region`, `eu_group` | Geographic region; for EU countries also Nordic / Western / Southern / Eastern, used for regional comparisons |
| `income_group` | World Bank classification by income per person (High / Upper middle / Lower middle) |
| `is_eu`, `is_euro`, `is_oecd` | Membership flags, useful as dashboard filters (check them before presenting; memberships change) |
| `small_firm_sample` | The country's stock market is too small to reach ~50 companies; treat its firm-level results with caution |
| `lat`, `lon` | Capital-city coordinates, for maps |

In [3]:
countries = q("""
SELECT c.*, COALESCE(f.companies, 0) AS companies_total
FROM dim_country c
LEFT JOIN (SELECT iso3, COUNT(*) AS companies FROM dim_company GROUP BY 1) f USING (iso3)
ORDER BY c.region, c.name
""")
countries

,iso3,iso2,eurostat_code,name,region,eu_group,income_group,is_eu,is_euro,is_oecd,small_firm_sample,lat,lon,companies_total
0,ZAF,ZA,NaN,South Africa,Africa,NaN,Upper middle income,False,False,False,False,-25.746,28.187,100
1,CHN,CN,NaN,China,East Asia,NaN,Upper middle income,False,False,False,False,40.050,116.286,100
2,JPN,JP,NaN,Japan,East Asia,NaN,High income,False,False,True,False,35.670,139.770,100
3,KOR,KR,NaN,South Korea,East Asia,NaN,High income,False,False,True,False,37.532,126.957,99
4,AUT,AT,AT,Austria,Europe,Western,High income,True,True,True,False,48.220,16.380,123
5,BEL,BE,BE,Belgium,Europe,Western,High income,True,True,True,False,50.837,4.368,197
6,CZE,CZ,CZ,Czechia,Europe,Eastern,High income,True,False,True,True,50.088,14.421,39
7,DNK,DK,DK,Denmark,Europe,Nordic,High income,True,False,True,False,55.676,12.568,197
8,EST,EE,EE,Estonia,Europe,Eastern,High income,True,True,True,True,59.439,24.759,34
9,FIN,FI,FI,Finland,Europe,Nordic,High income,True,True,True,False,60.161,24.953,200


In [4]:
fig = px.scatter_geo(countries, lat="lat", lon="lon", color="region", size="companies_total",
                     hover_name="name", hover_data=["income_group", "companies_total"],
                     projection="natural earth", title="The 34 countries (bubble size = companies in the database)")
fig.update_layout(height=520, margin=dict(l=0, r=0, t=50, b=0))
fig.show()

In [5]:
# how the sample is split
pd.concat({
    "by region": countries.groupby("region").size(),
    "by income group": countries.groupby("income_group").size(),
    "EU / non-EU": countries.groupby(countries.is_eu.map({True: "EU", False: "non-EU"})).size(),
    "EU group": countries[countries.is_eu].groupby("eu_group").size(),
}).rename("countries").to_frame()

countries
by region       Africa                       1
                East Asia                    3
                Europe                      21
                Latin America                3
                North America                2
                Oceania                      1
                South Asia                   1
                Southeast Asia               2
by income group High income                 28
                Lower middle income          1
                Upper middle income          5
EU / non-EU     EU                          21
                non-EU                      13
EU group        Eastern                      7
                Nordic                       3
                Southern                     4
                Western                      7

## 3 · Country indicators
**`fact_country_year`** is a *long* table: one row = one country × one indicator × one period → one value.
Long means adding a new indicator never changes the table structure; wide tables are built from it when needed.

| Column | Meaning |
|---|---|
| `iso3` | Country |
| `indicator_id` | Which series, e.g. `WB:SL.UEM.TOTL.ZS`, or for Eurostat dataset + filters (see below) |
| `period` / `year` | `'2019'`, or `'2012-S1'` for semi-annual series; `year` is always the integer year |
| `value` | The number, in the indicator's own unit |
| `flag` | Eurostat quality flag (see section 5) |

### 3.1 The full indicator catalog
`dim_indicator` describes each series. The *meaning* and *how to read it* columns below are written by hand and are not stored in the database.

In [6]:
# plain-language meaning of every indicator: code -> (meaning, how to read it)
M = {
 # World Bank WDI
 "NY.GDP.MKTP.CD": ("Total value of all goods and services produced in a year, in current US dollars.", "Size of the economy. Large countries dominate; use per-person or % measures to compare."),
 "NY.GDP.PCAP.PP.KD": ("GDP per person, adjusted for price differences between countries (PPP), in constant 2021 international dollars.", "Living standard that is comparable across countries. Higher = richer."),
 "NY.GDP.MKTP.KD.ZG": ("Yearly % change in real (inflation-adjusted) GDP.", "Economic growth. Negative = recession."),
 "FP.CPI.TOTL.ZG": ("Yearly % rise in consumer prices.", "Inflation. High inflation raises costs and uncertainty for firms."),
 "NV.IND.MANF.ZS": ("Share of GDP produced by manufacturing.", "Industrial structure; high in Asia and Central Europe."),
 "NV.SRV.TOTL.ZS": ("Share of GDP produced by services.", "Industrial structure; high in rich, service-based economies."),
 "NE.TRD.GNFS.ZS": ("Exports plus imports of goods and services, as % of GDP.", "Openness to trade. Can exceed 100% for small hub economies (Luxembourg, Singapore)."),
 "BX.KLT.DINV.WD.GD.ZS": ("Net foreign direct investment inflows (foreign firms buying or building businesses), % of GDP.", "Attractiveness to foreign investors. Can be distorted by tax-driven flows (Ireland, Luxembourg)."),
 "TX.VAL.TECH.MF.ZS": ("Share of manufactured exports that are high-technology (aerospace, computers, pharmaceuticals, ...).", "Technological level of industry."),
 "FS.AST.PRVT.GD.ZS": ("Credit that banks and other financial institutions give to companies and households, % of GDP.", "Depth of the financial system. Higher = easier access to finance."),
 "FR.INR.LEND": ("Interest rate banks charge for loans to the private sector, %.", "Cost of borrowing. Missing for many euro-area countries."),
 "CM.MKT.LCAP.GD.ZS": ("Total stock-market value of listed domestic companies, % of GDP.", "Size of the stock market relative to the economy (an outcome-side measure)."),
 "CM.MKT.LDOM.NO": ("Number of domestic companies listed on the stock exchange.", "Breadth of the stock market. Sets the ceiling for how many firms we can sample."),
 "IC.BUS.NDNS.ZS": ("New limited-liability companies registered per 1,000 people aged 15–64.", "Business dynamism / entrepreneurship."),
 "GC.TAX.TOTL.GD.ZS": ("Government tax revenue, % of GDP.", "Size of the tax burden."),
 "SL.UEM.TOTL.ZS": ("Share of the labour force without work and looking for it (ILO model), %.", "Labour-market slack. Lower = tighter labour market."),
 "SL.UEM.1524.ZS": ("Same, for ages 15–24.", "Youth unemployment is usually 2–3× the total rate."),
 "SL.TLF.CACT.ZS": ("Share of the population aged 15+ that is working or looking for work, %.", "Labour supply."),
 "SL.EMP.SELF.ZS": ("Self-employed as % of all employed people.", "High values usually mean more informal or less developed labour markets."),
 "SP.POP.TOTL": ("Total population.", "Market size. Level, not a ratio."),
 "SP.POP.1564.TO.ZS": ("Population aged 15–64, % of total.", "Working-age share."),
 "SP.POP.65UP.TO.ZS": ("Population aged 65+, % of total.", "Ageing; a pressure on public finances and labour supply."),
 "SP.URB.TOTL.IN.ZS": ("Share of the population living in cities, %.", "Urbanisation."),
 "SP.DYN.LE00.IN": ("Average years a newborn is expected to live.", "Health and development level."),
 "SI.POV.GINI": ("Income inequality, 0 (everyone earns the same) to 100 (one person earns everything).", "Higher = more unequal. Sparse: only survey years exist, so use as a snapshot."),
 "SE.TER.ENRR": ("People enrolled in tertiary education as % of the age group (can exceed 100).", "Skills supply."),
 "SE.XPD.TOTL.GD.ZS": ("Government spending on education, % of GDP.", "Public investment in skills."),
 "GB.XPD.RSDV.GD.ZS": ("Research and development spending (public and private), % of GDP.", "Innovation effort."),
 "IP.PAT.RESD": ("Patent applications filed by residents (a count).", "Innovation output. A count, so large countries dominate."),
 "IT.NET.USER.ZS": ("Share of the population using the internet, %.", "Digital readiness of customers and workers."),
 "IT.CEL.SETS.P2": ("Mobile subscriptions per 100 people (can exceed 100).", "Connectivity."),
 # WGI (all share the same scale)
 "GOV_WGI_GE.EST": ("Perceived quality of public services, civil service and policy implementation.", "Scale about −2.5 to +2.5, 0 ≈ world average; higher = better."),
 "GOV_WGI_RQ.EST": ("Perceived ability of the government to design policies that allow private business to develop.", "Scale about −2.5 to +2.5; higher = better."),
 "GOV_WGI_RL.EST": ("Confidence in contract enforcement, property rights, police and courts.", "Scale about −2.5 to +2.5; higher = better."),
 "GOV_WGI_CC.EST": ("Perceived extent to which public power is used for private gain, including petty and grand corruption.", "Scale about −2.5 to +2.5; higher = LESS corruption."),
 "GOV_WGI_VA.EST": ("Citizens' ability to take part in choosing the government; freedom of expression and media.", "Scale about −2.5 to +2.5; higher = more voice."),
 "GOV_WGI_PV.EST": ("Likelihood of political instability or politically motivated violence.", "Scale about −2.5 to +2.5; higher = MORE stable."),
 # IMF
 "NGDP_RPCH": ("Yearly % change in real GDP (IMF estimate; same idea as the World Bank series).", "Growth. Overlaps with WB:NY.GDP.MKTP.KD.ZG; the two sources can differ slightly."),
 "PCPIPCH": ("Yearly % change in average consumer prices (IMF).", "Inflation. Overlaps with the World Bank series."),
 "LUR": ("Unemployment rate (IMF).", "Overlaps with the World Bank / Eurostat series."),
 "GGXWDG_NGDP": ("Total government debt as % of GDP.", "Fiscal health. High debt can crowd out private borrowing and raise taxes."),
 "BCA_NGDPD": ("Current account balance as % of GDP: trade plus income flows with the rest of the world.", "Positive = the country earns more abroad than it spends (surplus)."),
 "NGDPDPC": ("GDP per person in current US dollars (not PPP-adjusted).", "Compare with the PPP measure: market exchange rates move with currency swings."),
}
# Eurostat series: "dataset:key" -> (meaning, how to read it)
E = {
 "nama_10_pc:CP_EUR_HAB": ("GDP per inhabitant in current euro.", "Level of output per person in euro."),
 "nama_10_pc:CP_PPS_EU27_2020_HAB": ("GDP per inhabitant in purchasing power standards (PPS), an artificial currency that removes price-level differences between EU countries.", "Best Eurostat measure for comparing living standards."),
 "une_rt_a:Y15-74": ("Unemployment rate, ages 15–74, % of the active population (labour force).", "Labour-market slack."),
 "une_rt_a:Y15-24": ("Unemployment rate, ages 15–24, % of the active population.", "Youth unemployment."),
 "prc_hicp_aind:CP00": ("Harmonised consumer-price inflation (HICP), annual average % change, all items.", "EU-comparable inflation."),
 "lc_lci_lev:": ("Hourly labour cost in euro for the business economy (wages plus employer social contributions).", "Direct operating cost of labour. Only years 2012, 2016, 2020–2025 exist."),
 "earn_mw_cur:EUR": ("Statutory monthly minimum wage in euro.", "Cost floor. Semi-annual. Missing where no statutory minimum exists (Austria, Denmark, Finland, Italy, Sweden)."),
 "earn_mw_cur:PPS": ("Statutory monthly minimum wage in PPS (price-level adjusted).", "Real purchasing power of the minimum wage."),
 "rd_e_gerdtot:EUR_HAB": ("Total R&D spending (business, government, universities) per inhabitant, euro.", "Innovation effort per person."),
 "rd_e_gerdtot:PC_GDP": ("Total R&D spending as % of GDP.", "Innovation intensity (EU target: 3%)."),
 "isoc_ci_ifp_iu:I_IU3": ("Share of individuals who used the internet in the last 3 months, %.", "Digital readiness of consumers."),
 "isoc_eb_ai:E_AI_TANY": ("Share of enterprises with 10+ employees using at least one AI technology, %.", "Firm digitalisation. Only 2021 and 2023–2025 exist."),
 "sbs_na_sca_r2:V11110": ("Number of enterprises in the business economy (excl. financial). LEGACY series, ends 2020.", "Business-population size. Not comparable with the 2021+ series."),
 "sbs_na_sca_r2:V12110": ("Turnover in million euro. LEGACY series, ends 2020.", "Business volume."),
 "sbs_na_sca_r2:V12150": ("Value added at factor cost in million euro. LEGACY series, ends 2020.", "Output created by businesses."),
 "sbs_na_sca_r2:V16110": ("Persons employed. LEGACY series, ends 2020.", "Employment in businesses."),
 "sbs_na_sca_r2:V91110": ("Apparent labour productivity: value added per person employed, thousand euro. LEGACY.", "Productivity. Higher = more output per worker."),
 "sbs_na_sca_r2:V92110": ("Gross operating rate: gross operating surplus as % of turnover. LEGACY.", "Rough profitability of the business sector."),
 "sbs_sc_ovw:ENT_NR": ("Number of enterprises (industry, construction, market services). CURRENT series, starts 2021.", "Business-population size. Broader than the legacy series."),
 "sbs_sc_ovw:EMP_NR": ("Persons employed. CURRENT series.", "Employment in businesses."),
 "sbs_sc_ovw:NETTUR_MEUR": ("Net turnover in million euro. CURRENT series.", "Business volume."),
 "sbs_sc_ovw:AV_MEUR": ("Value added in million euro. CURRENT series.", "Output created by businesses."),
 "sbs_sc_ovw:GOR_PC": ("Gross operating rate: gross operating surplus as % of turnover. CURRENT series.", "Rough profitability of the business sector."),
 "sbs_sc_ovw:AV_SAL_TEUR": ("Value added per employee, thousand euro. CURRENT series.", "Productivity."),
 "bd_size:ENT_NR": ("Number of active enterprises (business demography).", "Business population."),
 "bd_size:ENT_BRTH_NR": ("Number of enterprise births (new businesses) in the year.", "Entrepreneurship."),
 "bd_size:ENT_DTH_NR": ("Number of enterprise deaths (closures) in the year.", "Business exits."),
 "bd_size:ENT_BRTHR_PC": ("Birth rate: births divided by active enterprises, %.", "Business dynamism: higher = more new firms."),
 "bd_size:ENT_DTHR_PC": ("Death rate: deaths divided by active enterprises, %.", "Business turnover."),
 "nasa_10_ki:B2G_B3G_RAT_S11": ("Gross profit share of non-financial corporations: gross operating surplus / gross value added, %.", "How much of the value firms create is kept as profit (vs paid as wages). A country-level performance measure."),
 "nasa_10_ki:ROCE_S11": ("Gross return on capital employed (before tax) of non-financial corporations, %.", "Country-level return on capital. Not available for Czechia, Germany and Romania."),
 "nasa_10_ki:ROE_S11": ("Net return on equity (after tax) of non-financial corporations, %.", "Country-level profitability of shareholders' money."),
 "nasa_10_ki:IRG_S11": ("Gross investment rate of non-financial corporations: investment / value added, %.", "How much firms reinvest."),
 "nasa_10_ki:DIR_S11": ("Net debt-to-income ratio of non-financial corporations, %.", "Corporate leverage."),
}
MAIN_DIM = {"nama_10_pc": "unit", "une_rt_a": "age", "prc_hicp_aind": None, "lc_lci_lev": None, "earn_mw_cur": "currency",
            "rd_e_gerdtot": "unit", "isoc_ci_ifp_iu": "indic_is", "isoc_eb_ai": "indic_is", "sbs_na_sca_r2": "indic_sb",
            "sbs_sc_ovw": "indic_sbs", "bd_size": "indic_sbs", "nasa_10_ki": "na_item"}

import json
ind = q("SELECT indicator_id, source_id, source_code, name, theme, unit, snapshot_only, filters FROM dim_indicator ORDER BY source_id, source_code, indicator_id")
def ekey(r):
    if r.source_id != "ESTAT":
        return r.source_code
    f = r.filters; d = MAIN_DIM[r.source_code]
    return f"{r.source_code}:{f[d] if d else ('CP00' if r.source_code == 'prc_hicp_aind' else '')}"
ind["key"] = ind.apply(ekey, axis=1)
ind["meaning"] = ind.apply(lambda r: (E if r.source_id == "ESTAT" else M).get(r.key, ("", ""))[0], axis=1)
ind["how_to_read"] = ind.apply(lambda r: (E if r.source_id == "ESTAT" else M).get(r.key, ("", ""))[1], axis=1)
ind["unit"] = ind.unit.fillna("(see name)")
ind["short"] = np.where(ind.source_id == "ESTAT", ind.key, ind.source_id + ":" + ind.source_code)
missing = ind[ind.meaning == ""]
print("indicators:", len(ind), "| without a written meaning:", len(missing))
ind[["indicator_id", "theme", "name", "meaning", "how_to_read"]].assign(indicator_id=lambda d: d.indicator_id.str.slice(0, 48))

indicators: 77 | without a written meaning: 0


,indicator_id,theme,name,meaning,how_to_read
0,ESTAT:bd_size:age=TOTAL|indic_sbs=ENT_BRTHR_PC|n,business_demography,Business demography by size class and NACE Rev. 2 activity,"Birth rate: births divided by active enterprises, %.",Business dynamism: higher = more new firms.
1,ESTAT:bd_size:age=TOTAL|indic_sbs=ENT_BRTH_NR|na,business_demography,Business demography by size class and NACE Rev. 2 activity,Number of enterprise births (new businesses) in the year.,Entrepreneurship.
2,ESTAT:bd_size:age=TOTAL|indic_sbs=ENT_DTHR_PC|na,business_demography,Business demography by size class and NACE Rev. 2 activity,"Death rate: deaths divided by active enterprises, %.",Business turnover.
3,ESTAT:bd_size:age=TOTAL|indic_sbs=ENT_DTH_NR|nac,business_demography,Business demography by size class and NACE Rev. 2 activity,Number of enterprise deaths (closures) in the year.,Business exits.
4,ESTAT:bd_size:age=TOTAL|indic_sbs=ENT_NR|nace_r2,business_demography,Business demography by size class and NACE Rev. 2 activity,Number of active enterprises (business demography).,Business population.
5,ESTAT:earn_mw_cur:currency=EUR,labour_cost,Monthly minimum wages - bi-annual data,Statutory monthly minimum wage in euro.,"Cost floor. Semi-annual. Missing where no statutory minimum exists (Austria, Denmark, Finland, Italy, Swed..."
6,ESTAT:earn_mw_cur:currency=PPS,labour_cost,Monthly minimum wages - bi-annual data,Statutory monthly minimum wage in PPS (price-level adjusted).,Real purchasing power of the minimum wage.
7,ESTAT:isoc_ci_ifp_iu:ind_type=IND_TOTAL|indic_is,digital,Individuals - internet use,"Share of individuals who used the internet in the last 3 months, %.",Digital readiness of consumers.
8,ESTAT:isoc_eb_ai:indic_is=E_AI_TANY|nace_r2=C10-,digital,Artificial intelligence by size class of enterprise,"Share of enterprises with 10+ employees using at least one AI technology, %.",Firm digitalisation. Only 2021 and 2023–2025 exist.
9,ESTAT:lc_lci_lev:lcstruct=D1_D4_MD5|nace_r2=B-S_,labour_cost,Labour cost levels by NACE Rev. 2 activity,Hourly labour cost in euro for the business economy (wages plus employer social contributions).,"Direct operating cost of labour. Only years 2012, 2016, 2020–2025 exist."


In [7]:
# how many indicators per source and theme
px.bar(ind.groupby(["theme", "source_id"]).size().reset_index(name="indicators"), x="theme", y="indicators",
       color="source_id", title="Indicators by theme and source", height=380).update_xaxes(categoryorder="total descending")

### 3.2 Decoding the Eurostat series
A Eurostat dataset holds many series, selected by *dimension values* (`filters` in `dim_indicator`). Each (dataset + filters) is one indicator, so there are no duplicate rows. Common codes:

| Code | Meaning |
|---|---|
| `unit=PC_ACT` | % of the active population (labour force) |
| `unit=CP_EUR_HAB` / `CP_PPS_EU27_2020_HAB` | current prices, euro / PPS per inhabitant |
| `unit=PC_GDP`, `EUR_HAB` | % of GDP, euro per inhabitant |
| `unit=PC_IND`, `PC_ENT` | % of individuals, % of enterprises |
| `age=Y15-74`, `Y15-24` | age group of the unemployment rate |
| `nace_r2=B-S_X_O_S94` | industry, construction and market services (excl. public administration, defence, social security and membership organisations) |
| `nace_r2=B-N_S95_X_K` | business economy excluding financial and insurance activities |
| `nace_r2=B-S_X_O` | business economy, whole |
| `size_emp=GE10` / `TOTAL` | enterprises with 10+ employees / all sizes |
| `indic_sbs`, `indic_sb` | which business statistic (enterprises, turnover, births, ...) |
| `sector=S11` | non-financial corporations (national accounts) |

In [8]:
q("""
SELECT i.source_code AS dataset, COUNT(DISTINCT i.indicator_id) AS series,
       COUNT(DISTINCT f.iso3) AS countries, MIN(f.year) AS first_year, MAX(f.year) AS last_year,
       COUNT(DISTINCT f.year) AS distinct_years, COUNT(*) AS values
FROM dim_indicator i JOIN fact_country_year f USING (indicator_id)
WHERE i.source_id = 'ESTAT' GROUP BY 1 ORDER BY 1
""")

,dataset,series,countries,first_year,last_year,distinct_years,values
0,bd_size,5,21,2015,2024,10,476
1,earn_mw_cur,2,16,2012,2026,15,948
2,isoc_ci_ifp_iu,1,21,2012,2025,14,293
3,isoc_eb_ai,1,21,2021,2025,4,84
4,lc_lci_lev,1,21,2012,2025,8,167
5,nama_10_pc,2,21,2012,2025,14,588
6,nasa_10_ki,5,21,2012,2025,14,1311
7,prc_hicp_aind,1,21,2012,2025,14,294
8,rd_e_gerdtot,2,21,2012,2024,13,546
9,sbs_na_sca_r2,6,21,2012,2020,9,1132


### 3.3 Coverage: where do we actually have data?
Share of the years 2012–2024 with a value, per country and indicator. **Blank/dark cells are gaps.** Gaps are expected: recent years lag, and some indicators are simply not published for some countries.

In [9]:
cov = q("SELECT c.iso3, c.indicator_id, c.indicator_name, c.pct_years, i.source_id, i.source_code FROM v_coverage c JOIN dim_indicator i USING (indicator_id)")
cov = cov.merge(ind[["indicator_id", "short", "theme"]], on="indicator_id")
core = cov[cov.source_id != "ESTAT"].pivot(index="iso3", columns="short", values="pct_years")
order = ind[(ind.source_id != "ESTAT")].sort_values(["theme", "short"]).short.tolist()
core = core[order]
fig = px.imshow(core, aspect="auto", color_continuous_scale="Blues", zmin=0, zmax=100,
                labels=dict(color="% years"), title="Coverage, World Bank / WGI / IMF (all 34 countries)")
fig.update_layout(height=760, xaxis_tickangle=-60)
fig.show()
print("Least covered indicators (average % of years):")
core.mean().sort_values().head(8).round(1)

Least covered indicators (average % of years):


short
WB:FR.INR.LEND         43.000
WB:CM.MKT.LCAP.GD.ZS   70.800
WB:CM.MKT.LDOM.NO      71.500
WB:IP.PAT.RESD         75.500
WB:SE.XPD.TOTL.GD.ZS   75.600
WB:SI.POV.GINI         77.100
WB:GB.XPD.RSDV.GD.ZS   88.500
WB:SE.TER.ENRR         91.800
dtype: float64

In [10]:
eu = cov[cov.source_id == "ESTAT"].pivot(index="iso3", columns="short", values="pct_years")
fig = px.imshow(eu, aspect="auto", color_continuous_scale="Greens", zmin=0, zmax=100,
                labels=dict(color="% years"), title="Coverage, Eurostat (21 EU countries; sparse by design)")
fig.update_layout(height=620, xaxis_tickangle=-60)
fig.show()

### 3.4 Snapshot: the latest value for key indicators
`v_country_latest` gives the most recent non-null value per country and indicator. Compare countries here, but remember that latest years differ (shown in the year columns of the view).

In [11]:
KEY = {"WB:NY.GDP.PCAP.PP.KD": "GDP/person PPP ($)", "WB:NY.GDP.MKTP.KD.ZG": "GDP growth %", "WB:FP.CPI.TOTL.ZG": "Inflation %",
       "WB:SL.UEM.TOTL.ZS": "Unemployment %", "WB:SP.POP.TOTL": "Population", "WB:NE.TRD.GNFS.ZS": "Trade % GDP",
       "WB:GB.XPD.RSDV.GD.ZS": "R&D % GDP", "WB:IT.NET.USER.ZS": "Internet users %", "WB:CM.MKT.LCAP.GD.ZS": "Mkt cap % GDP",
       "WGI:GOV_WGI_RL.EST": "Rule of law", "WGI:GOV_WGI_GE.EST": "Gov. effectiveness", "WGI:GOV_WGI_CC.EST": "Corruption ctrl",
       "IMF:GGXWDG_NGDP": "Gov. debt % GDP"}
latest = q("SELECT iso3, indicator_id, value, year FROM v_country_latest")
snap = latest[latest.indicator_id.isin(KEY)].pivot(index="iso3", columns="indicator_id", values="value")[list(KEY)].rename(columns=KEY)
snap = countries.set_index("iso3")[["name"]].join(snap).sort_values("GDP/person PPP ($)", ascending=False)
snap

,name,GDP/person PPP ($),GDP growth %,Inflation %,Unemployment %,Population,Trade % GDP,R&D % GDP,Internet users %,Mkt cap % GDP,Rule of law,Gov. effectiveness,Corruption ctrl,Gov. debt % GDP
iso3,,,,,,,,,,,,,,
SGP,Singapore,"134,548.764",5.344,2.390,2.737,"6,036,860.000",313.299,1.808,94.378,111.303,1.474,2.111,1.970,166.000
LUX,Luxembourg,"128,475.284",0.359,2.051,6.400,"677,012.000",351.268,1.070,98.761,44.878,1.737,1.950,2.001,26.300
IRL,Ireland,"118,833.312",2.599,2.113,4.300,"5,395,790.000",246.170,1.588,97.188,27.832,1.538,1.610,1.556,38.300
USA,United States,"75,698.177",2.793,2.950,4.022,"340,003,797.000",25.016,3.447,94.694,212.252,0.997,1.302,1.079,122.300
DNK,Denmark,"71,034.579",3.478,1.372,5.358,"5,976,992.000",131.765,3.048,99.769,NaN,1.977,1.890,2.280,30.500
NLD,Netherlands,"70,493.905",1.074,3.348,3.700,"17,993,485.000",153.956,2.267,97.007,129.694,1.718,1.770,1.917,43.700
AUT,Austria,"63,787.989",-0.659,2.938,5.200,"9,177,982.000",108.826,3.255,94.920,22.585,1.643,1.504,1.304,79.200
BEL,Belgium,"63,311.277",1.098,3.143,5.700,"11,858,610.000",159.023,3.272,95.776,59.173,1.276,1.281,1.457,103.900
DEU,Germany,"62,654.602",-0.496,2.256,3.400,"83,516,593.000",79.091,3.154,93.500,43.628,1.608,1.506,1.764,62.200


In [12]:
# the same table as a heatmap: each column standardised (z-score) so very different units can share one colour scale
vals = snap.drop(columns=["name", "Population"])
z = (vals - vals.mean()) / vals.std()
fig = go.Figure(go.Heatmap(z=z.values, x=z.columns, y=snap.name, text=vals.round(1).values, texttemplate="%{text}",
                           colorscale="RdBu", zmid=0, colorbar=dict(title="z-score")))
fig.update_layout(title="Key indicators, latest values (colour = relative to the 34-country average; numbers = actual value)",
                  height=900, xaxis_tickangle=-40, yaxis_autorange="reversed")
fig.show()

### 3.5 Trends: explore an indicator over time
Pick an indicator from the dropdown; each line is a country (double-click a legend entry to isolate it).

In [13]:
TREND = {"NY.GDP.MKTP.KD.ZG": "WB:NY.GDP.MKTP.KD.ZG", "FP.CPI.TOTL.ZG": "WB:FP.CPI.TOTL.ZG", "SL.UEM.TOTL.ZS": "WB:SL.UEM.TOTL.ZS",
         "NY.GDP.PCAP.PP.KD": "WB:NY.GDP.PCAP.PP.KD", "GOV_WGI_RL.EST (rule of law)": "WGI:GOV_WGI_RL.EST",
         "IT.NET.USER.ZS": "WB:IT.NET.USER.ZS", "GGXWDG_NGDP (gov. debt)": "IMF:GGXWDG_NGDP", "GB.XPD.RSDV.GD.ZS (R&D)": "WB:GB.XPD.RSDV.GD.ZS"}
ts = q("SELECT iso3, indicator_id, year, value FROM fact_country_year WHERE indicator_id IN (" + ",".join(f"'{v}'" for v in TREND.values()) + ")")
names = countries.set_index("iso3").name
fig = go.Figure()
spans = []
for label, iid in TREND.items():
    d = ts[ts.indicator_id == iid]
    start = len(fig.data)
    for iso, g in d.groupby("iso3"):
        g = g.sort_values("year")
        fig.add_trace(go.Scatter(x=g.year, y=g.value, name=names[iso], mode="lines", visible=(label == list(TREND)[0]),
                                 hovertemplate=names[iso] + " %{x}: %{y:.2f}<extra></extra>"))
    spans.append((label, start, len(fig.data)))
buttons = [dict(label=l, method="update", args=[{"visible": [s <= i < e for i in range(len(fig.data))]}, {"title": l}]) for l, s, e in spans]
fig.update_layout(updatemenus=[dict(buttons=buttons, x=0, y=1.15, xanchor="left")], title=list(TREND)[0], height=520)
fig.show()

### 3.6 A known data break: Eurostat business statistics, 2020 → 2021
The enterprise count comes from two different tables. The legacy table ends in **2020**; the current one starts in **2021** with a wider scope and different rules. **Levels jump about 33% between 2020 and 2021 in the median country, which is not real growth.** There is no overlapping year, so the two cannot be converted into one series. For now they are stored as separate indicators and should not be joined into a single time line.

In [14]:
brk = q("""
SELECT f.iso3, f.year, f.value, CASE WHEN i.source_code = 'sbs_na_sca_r2' THEN 'legacy (to 2020)' ELSE 'current (from 2021)' END AS series
FROM fact_country_year f JOIN dim_indicator i USING (indicator_id)
WHERE i.indicator_id LIKE '%V11110%' OR (i.source_code = 'sbs_sc_ovw' AND i.filters::VARCHAR LIKE '%ENT_NR%')
""")
brk = brk[brk.iso3.isin(["SWE", "FRA", "DEU", "ITA", "POL", "PRT"])].assign(country=lambda d: d.iso3.map(names))
fig = px.line(brk, x="year", y="value", color="country", line_dash="series", markers=True, log_y=True,
              title="Number of enterprises: the series changes in 2021 (log scale)")
fig.add_vline(x=2020.5, line_dash="dot", line_color="grey")
fig.update_layout(height=460)
fig.show()

## 4 · Companies
### 4.1 Who is in the database (`dim_company`)
One row per company. `company_id` is `YF:<ticker>` (Yahoo Finance) or `ESEF:<lei>` (the Legal Entity Identifier used in EU filings).
The company's country is its **headquarters**, which is not necessarily where it earns its money (a Swiss-based multinational still counts as Swiss).

Sample selection: for each country we took listed companies, verified that Yahoo reports that country as headquarters (foreign cross-listings removed), and spread them across sectors by market cap.
**The same company can appear once from each source** (for example a Swedish bank in both Yahoo and ESEF); these are not matched yet.

In [15]:
q("SELECT * FROM dim_company ORDER BY random() LIMIT 8")

,company_id,source_id,source_key,name,iso3,sector,industry,exchange,currency,financial_currency,market_cap,employees
0,YF:BELI.JK,YF,BELI.JK,PT Global Digital Niaga Tbk,IDN,Consumer Cyclical,Internet Retail,JKT,IDR,IDR,"34,853,621,858,304.000","5,070.000"
1,YF:006125.KS,YF,006125.KS,"SK Discovery Co., Ltd.",KOR,Energy,Oil & Gas Refining & Marketing,KSC,KRW,KRW,"796,192,800,768.000",NaN
2,ESEF:5493008JPA4HYMH1HX51,ESEF,5493008JPA4HYMH1HX51,SES,LUX,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,YF:UNVR.JK,YF,UNVR.JK,PT Unilever Indonesia Tbk,IDN,Consumer Defensive,Household & Personal Products,JKT,IDR,IDR,"59,820,442,910,720.000","2,817.000"
4,YF:8YZ.SI,YF,8YZ.SI,Yangzijiang Maritime Development Ltd.,SGP,Financial Services,Asset Management,SES,SGD,USD,"2,022,844,288.000",NaN
5,YF:BLCO.TO,YF,BLCO.TO,Bausch + Lomb Corporation,CAN,Healthcare,Medical Instruments & Supplies,TOR,CAD,USD,"8,721,975,296.000","13,000.000"
6,ESEF:5UMCZOEYKCVFAW8ZLO05,ESEF,5UMCZOEYKCVFAW8ZLO05,ΕΘΝΙΚΗ ΤΡΑΠΕΖΑ ΤΗΣ ΕΛΛΑΔΟΣ Α.Ε.,GRC,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,YF:9766.T,YF,9766.T,Konami Group Corporation,JPN,Communication Services,Electronic Gaming & Multimedia,JPX,JPY,JPY,"3,010,031,517,696.000","5,186.000"


In [16]:
cc = q("""
SELECT c.name AS country, c.small_firm_sample AS small_sample, y.source_id, y.companies
FROM v_company_coverage y JOIN dim_country c USING (iso3)
""")
tot = cc.groupby("country").companies.sum().sort_values()
fig = px.bar(cc, y="country", x="companies", color="source_id", orientation="h", category_orders={"country": tot.index.tolist()},
             title="Companies with usable data (assets and net income) by country and source", height=820)
fig.add_vline(x=100, line_dash="dot", annotation_text="100", line_color="grey")
fig.show()
print("Small-sample countries:", ", ".join(cc[cc.small_sample].country.unique()))

Small-sample countries: Czechia, Estonia, Ireland, Lithuania, Latvia


In [17]:
sect = q("""SELECT COALESCE(sector, 'No sector (ESEF)') AS sector, source_id, COUNT(*) AS companies FROM dim_company GROUP BY 1, 2""")
px.bar(sect, x="sector", y="companies", color="source_id", title="Companies by sector (sector comes from Yahoo; ESEF filings carry none)",
       height=420).update_xaxes(categoryorder="total descending")

### 4.2 The financial statements (`fact_company_financial`)
One row = one company × one statement line × one fiscal year → one value. It keeps **each source's own item names**.

| Column | Meaning |
|---|---|
| `statement` | `income` (what happened over the year), `balance` (what the company owns and owes at the year end), `cashflow` (cash moves), `other` (employees) |
| `item` | The line item, exactly as the source names it (Yahoo: `Total Revenue`; ESEF: `Revenue`) |
| `fiscal_year_start`, `period_end` | The fiscal year. For a balance-sheet item, `period_end` is the balance date |
| `fy_start_derived` | `FALSE` = the start was reported in the filing; `TRUE` = computed as end − 1 year + 1 day |
| `currency` | Currency of the amount. **Never compare raw amounts across currencies; use ratios.** |
| `filing_id` | ESEF filing the value came from (empty for Yahoo) |

Yahoo gives 231 different items (including many derived by Yahoo itself); ESEF gives the 21 standard IFRS items we extracted.

In [18]:
IT = {
 "Total Revenue": "Sales: everything the company sold in the year (Yahoo).", "Operating Revenue": "Sales from core operations (Yahoo).",
 "Operating Income": "Profit from running the business, before interest and tax.", "Net Income": "Profit after everything: interest, tax, one-offs.",
 "Net Income Common Stockholders": "Net profit attributable to ordinary shareholders.", "Pretax Income": "Profit before tax.",
 "Total Assets": "Everything the company owns (cash, buildings, receivables, ...).", "Stockholders Equity": "Assets minus liabilities: the shareholders' share.",
 "Common Stock Equity": "Equity belonging to ordinary shareholders.", "Total Liabilities Net Minority Interest": "Everything the company owes.",
 "Cash And Cash Equivalents": "Cash and near-cash.",
 "Revenue": "Sales (ESEF). Banks and insurers often report no `Revenue`.", "RevenueFromContractsWithCustomers": "Sales from customer contracts (alternative revenue tag).",
 "InterestRevenueExpense": "Net interest income (banks).", "GrossProfit": "Revenue minus direct costs of goods sold.",
 "ProfitLossFromOperatingActivities": "Operating profit (ESEF).", "ProfitLoss": "Net profit including minority shareholders (ESEF).",
 "ProfitLossAttributableToOwnersOfParent": "Net profit attributable to the parent company's shareholders (ESEF).",
 "ProfitLossBeforeTax": "Profit before tax.", "IncomeTaxExpenseContinuingOperations": "Income tax charge.",
 "DepreciationAndAmortisationExpense": "Yearly write-down of assets (non-cash cost).",
 "Assets": "Total assets (ESEF).", "CurrentAssets": "Assets expected to be turned into cash within a year.", "NoncurrentAssets": "Long-lived assets.",
 "Liabilities": "Total liabilities.", "CurrentLiabilities": "Debts due within a year.", "Equity": "Total equity (ESEF).",
 "EquityAttributableToOwnersOfParent": "Equity belonging to the parent's shareholders.", "CashAndCashEquivalents": "Cash and near-cash.",
 "CashFlowsFromUsedInOperatingActivities": "Cash generated by running the business.", "CashFlowsFromUsedInInvestingActivities": "Cash spent on (or received from) investments.",
 "CashFlowsFromUsedInFinancingActivities": "Cash from loans, share issues, dividends.", "NumberOfEmployees": "Employees at year end.", "AverageNumberOfEmployees": "Average employees.",
}
top = q("""
SELECT c.source_id, f.statement, f.item, COUNT(*) AS values, COUNT(DISTINCT f.company_id) AS companies
FROM fact_company_financial f JOIN dim_company c USING (company_id)
GROUP BY 1, 2, 3
""")
top["companies_pct"] = (100 * top.companies / top.groupby("source_id").companies.transform("max")).round(0)
top["meaning"] = top.item.map(IT).fillna("")
cat = top[top.meaning != ""].sort_values(["source_id", "statement", "companies"], ascending=[True, True, False])
cat[["source_id", "statement", "item", "companies", "meaning"]]

,source_id,statement,item,companies,meaning
81,ESEF,balance,Assets,1312,Total assets (ESEF).
146,ESEF,balance,CashAndCashEquivalents,1292,Cash and near-cash.
14,ESEF,balance,Equity,1288,Total equity (ESEF).
80,ESEF,balance,NoncurrentAssets,1155,Long-lived assets.
236,ESEF,balance,CurrentLiabilities,1118,Debts due within a year.
199,ESEF,balance,CurrentAssets,1111,Assets expected to be turned into cash within a year.
200,ESEF,balance,EquityAttributableToOwnersOfParent,995,Equity belonging to the parent's shareholders.
58,ESEF,balance,Liabilities,793,Total liabilities.
221,ESEF,cashflow,CashFlowsFromUsedInFinancingActivities,1294,"Cash from loans, share issues, dividends."
15,ESEF,cashflow,CashFlowsFromUsedInInvestingActivities,1291,Cash spent on (or received from) investments.


**Example: raw statements for one company.** Toyota's fiscal year ends in March, so each column is a year ending 31 March.

In [19]:
ex = q("""
SELECT statement, item, period_end, value / 1e9 AS billion_jpy FROM fact_company_financial
WHERE company_id = 'YF:7203.T' AND item IN ('Total Revenue', 'Operating Income', 'Net Income', 'Total Assets', 'Stockholders Equity', 'Cash And Cash Equivalents')
""")
ex.pivot_table(index=["statement", "item"], columns="period_end", values="billion_jpy").round(0)

period_end                           2023-03-31  2024-03-31  2025-03-31  2026-03-31
statement item                                                                     
balance   Cash And Cash Equivalents   7,517.000   9,412.000   8,982.000  12,660.000
          Stockholders Equity        28,339.000  34,221.000  35,925.000  39,919.000
          Total Assets               74,303.000  90,114.000  93,601.000 105,522.000
income    Net Income                  2,451.000   4,945.000   4,765.000   3,848.000
          Operating Income            2,725.000   5,353.000   4,796.000   3,766.000
          Total Revenue              37,154.000  45,095.000  48,037.000  50,685.000

### 4.3 Fiscal years: why `fiscal_year_start` matters
Companies close their books in different months, so "2023" means different 12 months for different firms. The heatmap shows the share of each country's company-years that end in each month: Japan and India cluster in March, Australia in June, and most of Europe in December.
`v_company_year` assigns each fiscal year to **the calendar year in which most of it falls**: a year running 1 Apr 2023 – 31 Mar 2024 counts as fiscal year 2023.

In [20]:
mm = q("""SELECT iso3, EXTRACT(month FROM period_end)::INT AS month, COUNT(*) AS n FROM v_company_year GROUP BY 1, 2""")
mm = mm.pivot(index="iso3", columns="month", values="n").fillna(0)
share = mm.div(mm.sum(axis=1), axis=0).mul(100).round(0)
share.index = share.index.map(names)
fig = px.imshow(share, aspect="auto", color_continuous_scale="Viridis", labels=dict(x="fiscal year end month", y="", color="% of company-years"),
                title="When do fiscal years end? Share of company-years by end month")
fig.update_layout(height=720)
fig.show()

### 4.4 Comparable performance metrics (`v_company_year`)
Because currencies and company sizes differ wildly, the analysis uses **ratios**. The view puts the standard metrics side by side for each company-year, mapping each source's item names (`company_metric_map`):

| Metric | Built from | Meaning |
|---|---|---|
| `revenue`, `operating_income`, `net_income`, `total_assets`, `equity` | the source items mapped in `company_metric_map` | amounts in the company's reporting currency |
| **`roa`** | net income ÷ total assets | **Return on assets: profit per unit of assets.** The main performance measure. ~3% is typical |
| `operating_margin` | operating income ÷ revenue | share of sales kept as operating profit |
| `revenue_growth` | revenue ÷ previous year's revenue − 1 | growth; only when the previous year ended about 12 months earlier |
| `fiscal_year` | midpoint of the fiscal year | the calendar year it is assigned to |

Ratios here are **raw, not winsorised**: one company with tiny assets can produce a ROA of 300%, so winsorise (cap at the 1st and 99th percentile) before analysis, as below.

In [21]:
q("""
SELECT name, iso3, sector, fiscal_year, fiscal_year_start, period_end, currency, revenue, net_income, total_assets, roa, operating_margin, revenue_growth
FROM v_company_year WHERE company_id IN ('YF:7203.T', 'YF:AAPL', 'YF:SAP.DE', 'YF:BHP.AX') ORDER BY name, period_end
""")

,name,iso3,sector,fiscal_year,fiscal_year_start,period_end,currency,revenue,net_income,total_assets,roa,operating_margin,revenue_growth
0,Apple Inc.,USA,Technology,2022,2021-10-01,2022-09-30,USD,"394,328,000,000.000","99,803,000,000.000","352,755,000,000.000",0.283,0.303,NaN
1,Apple Inc.,USA,Technology,2023,2022-10-01,2023-09-30,USD,"383,285,000,000.000","96,995,000,000.000","352,583,000,000.000",0.275,0.298,-0.028
2,Apple Inc.,USA,Technology,2024,2023-10-01,2024-09-30,USD,"391,035,000,000.000","93,736,000,000.000","364,980,000,000.000",0.257,0.315,0.020
3,Apple Inc.,USA,Technology,2025,2024-10-01,2025-09-30,USD,"416,161,000,000.000","112,010,000,000.000","359,241,000,000.000",0.312,0.320,0.064
4,BHP Group Limited,AUS,Basic Materials,2022,2022-07-01,2023-06-30,USD,"53,817,000,000.000","12,921,000,000.000","101,296,000,000.000",0.128,0.408,NaN
5,BHP Group Limited,AUS,Basic Materials,2023,2023-07-01,2024-06-30,USD,"55,658,000,000.000","7,897,000,000.000","102,362,000,000.000",0.077,0.412,0.034
6,BHP Group Limited,AUS,Basic Materials,2024,2024-07-01,2025-06-30,USD,"51,262,000,000.000","9,019,000,000.000","108,790,000,000.000",0.083,0.372,-0.079
7,BHP Group Limited,AUS,Basic Materials,2025,2025-07-01,2026-06-30,USD,"58,760,000,000.000","9,833,000,000.000","121,387,000,000.000",0.081,0.434,0.146
8,SAP SE,DEU,Technology,2022,2022-01-01,2022-12-31,EUR,"29,519,000,000.000","2,284,000,000.000","72,159,000,000.000",0.032,0.205,NaN
9,SAP SE,DEU,Technology,2023,2023-01-01,2023-12-31,EUR,"31,207,000,000.000","6,139,000,000.000","68,331,000,000.000",0.090,0.193,0.057


In [22]:
cy = q("SELECT * FROM v_company_year")
metrics = ["revenue", "operating_income", "net_income", "total_assets", "equity", "roa", "operating_margin", "revenue_growth"]
(cy.groupby("source_id")[metrics].agg(lambda s: s.notna().mean() * 100).round(0).T
   .rename(columns=lambda c: f"{c}: % rows with a value"))

source_id,ESEF: % rows with a value,YF: % rows with a value
revenue,58.000,100.000
operating_income,62.000,91.000
net_income,73.000,100.000
total_assets,79.000,100.000
equity,99.000,100.000
roa,72.000,100.000
operating_margin,53.000,89.000
revenue_growth,38.000,73.000


**What does ROA look like?** Distribution of company-year ROA, clipped to the 1st–99th percentile (winsorised) for display.

In [23]:
d = cy.dropna(subset=["roa"]).copy()
lo, hi = d.roa.quantile([0.01, 0.99])
d["roa_w"] = d.roa.clip(lo, hi)
print(f"n = {len(d):,} company-years | 1st / 99th percentile: {lo:.2f} / {hi:.2f} | median {d.roa.median():.3f} | raw min {d.roa.min():.1f}, raw max {d.roa.max():.1f}")
fig = px.histogram(d, x="roa_w", color="source_id", nbins=70, barmode="overlay", opacity=0.65,
                   labels={"roa_w": "ROA (winsorised)"}, title="Return on assets, all company-years")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=400)
fig.show()

n = 15,532 company-years | 1st / 99th percentile: -0.53 / 0.31 | median 0.034 | raw min -719.2, raw max 148.3


**Companies in different countries compared.** Distribution of (winsorised) ROA per country, ordered by median. Countries flagged ⚠ have a small company sample.

In [24]:
d["country"] = d.iso3.map(names)
small = set(countries[countries.small_firm_sample].name)
n_firms = d.groupby("country").company_id.nunique()
med = d.groupby("country").roa_w.median().sort_values()
d["label"] = d.country.map(lambda c: f"{c}{' ⚠' if c in small else ''} (n={n_firms[c]})")
order = [f"{c}{' ⚠' if c in small else ''} (n={n_firms[c]})" for c in med.index]
fig = px.box(d, y="label", x="roa_w", points=False, category_orders={"label": order}, labels={"roa_w": "ROA (winsorised)", "label": ""},
             title="ROA by country (box = middle 50% of company-years, line = median)")
fig.update_layout(height=900)
fig.show()

In [25]:
bys = d[d.sector.notna()].groupby("sector").agg(median_roa=("roa_w", "median"), company_years=("roa_w", "size"), companies=("company_id", "nunique")).sort_values("median_roa")
px.bar(bys.reset_index(), x="median_roa", y="sector", orientation="h", text=bys.companies.values, title="Median ROA by sector (Yahoo companies; label = number of companies)", height=420)

## 5 · Quality notes and caveats
What to keep in mind before reading anything into the data:

1. **Association, not causation.** Company performance depends mostly on the company and its sector. Country conditions explain a smaller part.
2. **Headquarters ≠ where the money is made.** Large listed firms earn much of their revenue abroad.
3. **Large listed firms only.** No small or private companies; survivorship bias (companies that disappeared are missing).
4. **Never compare raw amounts across currencies.** Use ratios (ROA, margin, growth) or convert explicitly.
5. **About 4 fiscal years per company** from Yahoo; ESEF goes back to 2019 for some.
6. **Different accounting standards**: ESEF = IFRS; Yahoo mixes IFRS, US GAAP and local standards.
7. **Two sources overlap.** The same company may appear in both Yahoo and ESEF.
8. **Small samples**: Estonia, Latvia, Lithuania, Czechia and Ireland have fewer than 50 firms because their markets are small.
9. **Banks and insurers** often have no `revenue` or `operating_income`, so margin and growth cover fewer firms than ROA.
10. **Recent years have gaps** because statistics are published with a delay.
11. **Eurostat business statistics have a series break between 2020 and 2021** (see 3.6) and its tables are not available for non-EU countries.
12. **Indicators that overlap** (growth, inflation and unemployment appear in World Bank, IMF and Eurostat) can differ slightly between sources; they are kept as separate indicators.

In [26]:
quality = {
  "Company-years with ROA computed": f"{cy.roa.notna().sum():,} of {len(cy):,}",
  "Companies with no sector (all ESEF)": f"{q('SELECT COUNT(*) FROM dim_company WHERE sector IS NULL').iloc[0, 0]:,}",
  "Financial rows with a derived (not reported) fiscal year start": f"{q('SELECT ROUND(100.0 * AVG(fy_start_derived::INT), 1) FROM fact_company_financial').iloc[0, 0]}%",
  "Distinct reporting currencies": f"{q('SELECT COUNT(DISTINCT currency) FROM fact_company_financial').iloc[0, 0]}",
  "Eurostat values carrying a quality flag": f"{q('SELECT COUNT(*) FROM fact_country_year WHERE flag IS NOT NULL').iloc[0, 0]:,} of {q('SELECT COUNT(*) FROM fact_country_year f JOIN dim_indicator i USING (indicator_id) WHERE i.source_id = \'ESTAT\'').iloc[0, 0]:,}",
}
pd.Series(quality, name="value").to_frame()

,value
Company-years with ROA computed,"15,532 of 17,608"
Companies with no sector (all ESEF),"1,315"
Financial rows with a derived (not reported) fiscal year start,93.3%
Distinct reporting currencies,26
Eurostat values carrying a quality flag,"518 of 6,930"


In [27]:
# Eurostat quality flags found in the data
FLAGS = {"b": "break in time series", "p": "provisional", "e": "estimated", "d": "definition differs", "u": "low reliability", "c": "confidential", "n": "not significant", "s": "Eurostat estimate"}
fl = q("SELECT flag, COUNT(*) AS values FROM fact_country_year WHERE flag IS NOT NULL GROUP BY 1 ORDER BY 2 DESC")
fl["meaning"] = fl.flag.map(lambda f: ", ".join(FLAGS.get(ch, ch) for ch in f))
fl

,flag,values,meaning
0,b,235,break in time series
1,p,163,provisional
2,e,66,estimated
3,d,28,definition differs
4,ep,16,"estimated, provisional"
5,be,6,"break in time series, estimated"
6,bp,2,"break in time series, provisional"
7,bd,2,"break in time series, definition differs"


## 6 · Data dictionary
Every table and view in the DuckDB file, with each column's type and meaning. The last cell checks that every column has a description.

In [28]:
D = {
 "dim_source": {"source_id": "Short code of the data source (WB, WGI, IMF, ESTAT, YF, ESEF).", "name": "Full name.", "base_url": "API address.", "retrieved_at": "When the raw files were downloaded."},
 "dim_country": {"iso3": "Country key (ISO 3166 alpha-3).", "iso2": "Two-letter code.", "eurostat_code": "Eurostat code (EL for Greece); empty outside the EU.", "name": "Country name.", "region": "Geographic region.",
                 "eu_group": "Nordic / Western / Southern / Eastern (EU only).", "income_group": "World Bank income class.", "is_eu": "EU member.", "is_euro": "Uses the euro.", "is_oecd": "OECD member.",
                 "small_firm_sample": "Stock market too small to reach ~50 firms.", "lat": "Capital latitude.", "lon": "Capital longitude."},
 "dim_indicator": {"indicator_id": "Series key, e.g. 'WB:SP.POP.TOTL'.", "source_id": "Source.", "source_code": "The source's own code (dataset code for Eurostat).", "name": "Indicator name from the source.",
                   "description": "Longer description (IMF) or filter summary (Eurostat).", "theme": "Topic group (economy, labour, governance, ...).", "unit": "Unit of the value, when the source gives one.",
                   "snapshot_only": "TRUE = sparse series; use only as a latest-available snapshot.", "filters": "Eurostat dimension values that define the series."},
 "fact_country_year": {"iso3": "Country.", "indicator_id": "Series.", "period": "'2019' or '2012-S1' (semi-annual).", "year": "Integer year.", "value": "The value, in the indicator's unit.", "flag": "Eurostat quality flag."},
 "dim_company": {"company_id": "'YF:<ticker>' or 'ESEF:<lei>'.", "source_id": "YF or ESEF.", "source_key": "The ticker or LEI.", "name": "Company name.", "iso3": "Headquarters country.",
                 "sector": "Yahoo sector (empty for ESEF).", "industry": "Yahoo industry.", "exchange": "Stock exchange.", "currency": "Trading currency.", "financial_currency": "Currency of the statements (Yahoo).",
                 "market_cap": "Market value when collected (in trading currency).", "employees": "Full-time employees."},
 "fact_company_financial": {"company_id": "Company.", "statement": "income / balance / cashflow / other.", "item": "Line item name as the source reports it.", "fiscal_year_start": "First day of the fiscal year.",
                            "period_end": "Last day of the fiscal year (balance date).", "fy_start_derived": "TRUE = start computed, FALSE = reported.", "currency": "Currency of the value.", "value": "Amount.", "filing_id": "ESEF filing id."},
 "company_metric_map": {"source_id": "Source.", "statement": "Statement.", "item": "Source item name.", "metric": "Standard metric it feeds.", "priority": "Lower wins when several items map to one metric."},
 "v_country_latest": {"iso3": "Country.", "indicator_id": "Series.", "indicator_name": "Name.", "theme": "Theme.", "period": "Period of the latest value.", "year": "Year of the latest value.", "value": "Latest non-null value."},
 "v_coverage": {"iso3": "Country.", "indicator_id": "Series.", "source_id": "Source.", "indicator_name": "Name.", "theme": "Theme.", "n_years": "Years 2012–2024 with a value.", "first_year": "Earliest year with data.",
                "last_year": "Latest year with data.", "pct_years": "n_years as % of the 13 years."},
 "v_company_year": {"company_id": "Company.", "source_id": "YF or ESEF.", "name": "Company name.", "iso3": "Headquarters country.", "sector": "Sector.", "fiscal_year": "Calendar year most of the fiscal year falls in.",
                    "fiscal_year_start": "Start of fiscal year.", "period_end": "End of fiscal year.", "currency": "Currency of the amounts.", "revenue": "Sales.", "operating_income": "Operating profit.",
                    "net_income": "Net profit.", "total_assets": "Total assets.", "equity": "Shareholders' equity.", "roa": "net_income / total_assets.", "operating_margin": "operating_income / revenue.",
                    "revenue_growth": "Revenue vs previous year."},
 "v_company_coverage": {"iso3": "Country.", "source_id": "YF or ESEF.", "companies": "Companies with net income and assets.", "firm_years": "Company-years with both.", "years_per_company": "Average years per company."},
}
cols = q("SELECT table_name, column_name, data_type FROM information_schema.columns WHERE table_schema = 'main' ORDER BY table_name, ordinal_position")
cols["meaning"] = [D.get(t, {}).get(c, "") for t, c in zip(cols.table_name, cols.column_name)]
rows = q("SELECT table_name, estimated_size AS rows FROM duckdb_tables()")
print("columns without a description:", (cols.meaning == "").sum())
cols.merge(rows, on="table_name", how="left")[["table_name", "rows", "column_name", "data_type", "meaning"]]

columns without a description: 0


,table_name,rows,column_name,data_type,meaning
0,company_metric_map,14,source_id,VARCHAR,Source.
1,company_metric_map,14,statement,VARCHAR,Statement.
2,company_metric_map,14,item,VARCHAR,Source item name.
3,company_metric_map,14,metric,VARCHAR,Standard metric it feeds.
4,company_metric_map,14,priority,BIGINT,Lower wins when several items map to one metric.
5,dim_company,3902,company_id,VARCHAR,'YF:<ticker>' or 'ESEF:<lei>'.
6,dim_company,3902,source_id,VARCHAR,YF or ESEF.
7,dim_company,3902,source_key,VARCHAR,The ticker or LEI.
8,dim_company,3902,name,VARCHAR,Company name.
9,dim_company,3902,iso3,VARCHAR,Headquarters country.


In [29]:
con.close()